# Message History

In [1]:
# LangSmith 추적 설정 부분
from dotenv import load_dotenv
import os

load_dotenv()

project_name = "wanted_2nd_prompt_basic"
os.environ["LANGSMITH_PROJECT"] = project_name

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

#--- 모델 설정 ---#
model = ChatOpenAI(
    temperature=0.1,
    model="gpt-4.1-mini",
    verbose=True
)

In [3]:
from typing import Dict
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

In [5]:
# 1. 프롬프트에 history 자리를 확보
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 AI 도우미야, 간략하게 응답하도록 해"),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{question}")
])
prompt

ChatPromptTemplate(input_variables=['history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchai

In [7]:
chain = prompt | model | StrOutputParser()
chain

ChatPromptTemplate(input_variables=['history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchai

In [8]:
# 2. 대화 내용 저장소 만들기
stores: Dict[str, InMemoryChatMessageHistory] = {} # 이 스토어는 딕셔너리인데 str과 InMemory~ 형태로 되어있다
def get_store(session_id: str):
    # 첫 대화일 때
    if session_id not in stores:
        stores[session_id] = InMemoryChatMessageHistory()
    return stores[session_id]

In [10]:
# 3. 히스토리랑 래핑
with_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: get_store(session_id),
    input_messages_key="question",
    history_messages_key="history"
)

In [12]:
cfg = {"configurable": {"session_id": "user-123"}}
result = with_history.invoke({
    "question": "내가 좋아하는 폰은 갤럭시야"
}, config=cfg
)
print(result)

갤럭시 폰 정말 인기 많죠! 어떤 모델을 좋아하세요?


In [13]:
result = with_history.invoke({
    "question": "장점만 요약해줘"
}, config=cfg
)
print(result)

갤럭시 폰 장점 요약:
- 뛰어난 디스플레이 품질 (AMOLED)
- 강력한 성능과 빠른 처리 속도
- 다양한 카메라 기능과 우수한 사진 품질
- 방수·방진 기능 지원
- 확장 가능한 저장 공간 (microSD 지원 모델)
- 삼성 생태계와의 높은 호환성


In [14]:
result = with_history.invoke({
    "question": "단점도 요약해줘"
}, config=cfg
)
print(result)

갤럭시 폰 단점 요약:
- 가격이 비교적 높음
- 소프트웨어 업데이트 지원 기간이 짧을 수 있음
- 일부 모델에서 배터리 사용 시간이 아쉬움
- 무거운 UI와 사전 설치 앱(블로트웨어) 존재
- 빠른 발열 현상 발생 가능성


In [15]:
result = with_history.invoke({
    "question": "너는 아이폰과 비교해서 뭐가 더 좋아?"
}, config=cfg
)
print(result)

갤럭시는 맞춤 설정과 확장성, 다양한 모델 선택지에서 강점이 있고, 아이폰은 소프트웨어 최적화와 보안, 긴 업데이트 지원이 뛰어납니다. 어떤 점을 더 중요하게 생각하느냐에 따라 다릅니다.


In [17]:
stores["user-123"].messages

[HumanMessage(content='내가 좋아하는 폰은 갤럭시야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='갤럭시 폰 정말 인기 많죠! 어떤 모델을 좋아하세요?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='장점만 요약해줘', additional_kwargs={}, response_metadata={}),
 AIMessage(content='갤럭시 폰 장점 요약:\n- 뛰어난 디스플레이 품질 (AMOLED)\n- 강력한 성능과 빠른 처리 속도\n- 다양한 카메라 기능과 우수한 사진 품질\n- 방수·방진 기능 지원\n- 확장 가능한 저장 공간 (microSD 지원 모델)\n- 삼성 생태계와의 높은 호환성', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='단점도 요약해줘', additional_kwargs={}, response_metadata={}),
 AIMessage(content='갤럭시 폰 단점 요약:\n- 가격이 비교적 높음\n- 소프트웨어 업데이트 지원 기간이 짧을 수 있음\n- 일부 모델에서 배터리 사용 시간이 아쉬움\n- 무거운 UI와 사전 설치 앱(블로트웨어) 존재\n- 빠른 발열 현상 발생 가능성', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='너는 아이폰과 비교해서 뭐가 더 좋아?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='갤럭시는 맞춤 설정과 확장성, 다양한 모델 선택지에서 강점이 있고, 아이폰은 소프트웨어 최적화와 보안, 긴 업데이트 지원이 뛰어납니다. 어떤 점을 더 중요하게 생각하느냐에 따라 다릅니다.', additional_kwargs={}, respo